# lab_18_lorentzian

Goal
----
Show the result the 1/f^2 formula hides: a real oscillator's carrier is NOT a
delta line and NOT actually 1/f^2 all the way in. Phase undergoes a random walk
(Wiener process), which makes the CARRIER AUTOCORRELATION decay exponentially,
so the spectrum is a LORENTZIAN with a finite 3-dB linewidth. The 1/f^2 skirt is
only the far-from-carrier asymptote; near the carrier the Lorentzian flattens
(finite peak) and total power is conserved (= carrier power).

Model
-----
  phi(t): Wiener process with Var[phi(t)] = 2 D t   (D = phase diffusion [rad^2/s])
  carrier x(t) = cos(2 pi f0 t + phi(t))
  R_x(tau) = 1/2 cos(2 pi f0 tau) e^{-D|tau|}  ->  S_x = Lorentzian
  one-sided about carrier:  S(df) ∝ D / (D^2 + (2 pi df)^2)
  HWHM(df) = D/(2 pi) Hz ;  FWHM (3-dB linewidth) = D/pi Hz.

Figure
------
  static/figures/lorentzian_carrier_lineshape.png

---

> 本 notebook 由 `scripts/make_notebooks.py` 從 `simulations/lab_18_lorentzian.py` **自動產生**（generated snapshot，非手寫檔）。
> 權威版本是 repo 裡的 lab script；lab 更新後請重跑產生器同步。
> 執行需求：clone [isf-teaching-site](https://github.com/gmcycle7/isf-teaching-site)（要 import `simulations/common`）＋ `numpy` / `scipy` / `matplotlib`。

In [ ]:
# --- Setup：本 notebook 需要教學網站 repo 的 simulations/common 模組 ---
# 還沒有原始碼的話，先 clone repo，並把本 notebook 放在 repo 目錄樹內執行：
#     git clone https://github.com/gmcycle7/isf-teaching-site.git
# 相依套件只有三個：pip install numpy scipy matplotlib（外加 jupyter 本身）
import sys
from pathlib import Path

def _find_repo_root():
    """從目前工作目錄往上找，直到看到 simulations/common 為止。"""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "simulations" / "common").is_dir():
            return base
    raise FileNotFoundError(
        "找不到 simulations/common —— 請把本 notebook 放進 isf-teaching-site "
        "repo 目錄樹內執行（git clone https://github.com/gmcycle7/isf-teaching-site.git），"
        "或手動把 <repo>/simulations/common 加入 sys.path")

ROOT = _find_repo_root()
for _p in (str(ROOT), str(ROOT / "simulations" / "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root:", ROOT)

# CJK 字型：圖的標籤有繁體中文；找不到 CJK 字型只影響文字顯示、不影響任何數值
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
_cjk = next((f for f in ["Heiti TC", "Arial Unicode MS", "STHeiti",
                         "Hiragino Sans GB", "Songti SC", "PingFang TC",
                         "Noto Sans CJK TC", "Microsoft JhengHei"]
             if f in _avail), None)
if _cjk:
    plt.rcParams["font.family"] = [_cjk, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False   # ASCII 減號，避免變方塊
print("CJK font:", _cjk or "(none found — 中文標籤可能顯示為方塊)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# notebook 版 savefig：改成 inline 顯示。
# （原始 lab script 的 plot_utils.savefig 會把 PNG 寫進 static/figures/ 且從不
#   show()；在 notebook 裡我們直接把圖畫在 cell 輸出。）
def savefig(fig, name, verbose=True):
    plt.show()
    plt.close(fig)

RNG = np.random.default_rng(18)

In [ ]:
def main():
    print("[lab_18] Lorentzian carrier lineshape ...")
    fs = 4096.0
    n = 2 ** 20
    t = np.arange(n) / fs
    f0 = 400.0                 # carrier (normalized units)
    D = 2.0                    # phase diffusion [rad^2/s] -> FWHM = D/pi ~ 0.64 Hz

    # Wiener phase: increments N(0, 2 D dt)
    dphi = RNG.standard_normal(n) * np.sqrt(2 * D / fs)
    phi = np.cumsum(dphi)
    x = np.cos(2 * np.pi * f0 * t + phi)

    f, P = welch(x, fs=fs, nperseg=2 ** 16, scaling="density")
    off = f - f0
    m = (np.abs(off) < 50) & (off != 0)

    # theory Lorentzian about carrier, normalized to the simulated peak region
    lor = D / (D ** 2 + (2 * np.pi * off) ** 2)
    # scale theory to sim near carrier
    near = np.abs(off) < 2
    scale = np.median(P[m & near]) / np.median(lor[m & near])
    lor_s = lor * scale

    fwhm = D / np.pi  # Hz

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

    # (a) lineshape: simulated vs Lorentzian vs 1/f^2 asymptote
    ax = axes[0]
    ax.loglog(off[off > 0], P[off > 0], color="tab:blue", lw=0.8, alpha=0.6,
              label="simulated $S(\\Delta f)$")
    pos = off > 0
    ax.loglog(off[pos], lor_s[pos], "k--", lw=1.5, label="Lorentzian 理論")
    # 1/f^2 asymptote: far-out Lorentzian ~ scale*D/(2 pi df)^2
    asym = scale * D / (2 * np.pi * off) ** 2
    ax.loglog(off[pos], asym[pos], color="tab:red", ls=":", lw=1.3,
              label="$1/\\Delta f^2$ 漸近（遠離載波）")
    ax.axvline(fwhm / 2, color="tab:green", lw=1, ls="-.",
               label=f"HWHM = D/2π = {fwhm/2:.2f} Hz")
    ax.set_xlim(0.05, 50)
    ax.set_xlabel("offset from carrier $\\Delta f$ (normalized)")
    ax.set_ylabel("$S(\\Delta f)$")
    ax.set_title("載波是 Lorentzian：近載波轉平（有限峰），$1/f^2$ 只是遠端漸近")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)

    # (b) phase variance grows linearly (random walk) -> exponential R_x
    ax = axes[1]
    # estimate Var[phi(t)] across many short segments
    seglen = 2000
    nseg = 300
    taus = np.arange(1, seglen) / fs
    var_acc = np.zeros(seglen - 1)
    cnt = 0
    for _ in range(nseg):
        s = RNG.integers(0, n - seglen)
        seg = phi[s:s + seglen] - phi[s]
        var_acc += seg[1:] ** 2
        cnt += 1
    var_phi = var_acc / cnt
    ax.plot(taus, var_phi, color="tab:blue", lw=1.2, label="量測 Var[Δφ(τ)]")
    ax.plot(taus, 2 * D * taus, "k--", lw=1.5, label="理論 $2D\\tau$（線性）")
    ax.set_xlabel("time lag $\\tau$ (s, normalized)")
    ax.set_ylabel("Var[$\\Delta\\phi(\\tau)$] [rad$^2$]")
    ax.set_title("相位是 random walk：方差線性成長 → 載波自相關指數衰減")
    ax.legend(fontsize=9)
    print(f"    D={D} rad^2/s -> FWHM linewidth = D/pi = {fwhm:.3f} Hz")
    savefig(fig, "lorentzian_carrier_lineshape.png")

In [ ]:
# 執行整個 lab（對應原 script 的 __main__）
main()